# Data Analysis and Visualization

Après avoir effectué l'ensemble de l'état de l'art, installé l'environnement pour le modèle de baseline Open-Unmix, et téléchargé la base de données, j'aimerai dans un premier temps faire de l'analyse exploratoire des données (EDA) sur la base de données MUSDB18. L'objectif est de mieux comprendre les caractéristiques des données audio, telles que la distribution des durées des morceaux, les niveaux de volume, et les fréquences dominantes. Ensuite, je vais visualiser ces caractéristiques à l'aide de graphiques appropriés pour identifier des tendances ou des anomalies potentielles dans les données.

#### Importation des bibliothèques nécessaires

In [ ]:
!pip install musdb librosa matplotlib pandas numpy seaborn
import musdb
import librosa
import librosa.display
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import os

#### Chargement de la base de données MUSDB18

In [ ]:
mus = musdb.DB(root='data/musdb18', subsets='train') 

In [ ]:
stats = []

for track in mus.tracks:
    track_dict = {
        'title': track.name,
        'duration_sec': track.duration,
        'sr': track.rate,
        'sources': list(track.sources.keys())
    }
    
    # Analyser chaque source
    for source_name, source_audio in track.sources.items():
        # source_audio: numpy array (samples, channels)
        audio = source_audio.T  # transpose pour avoir (channels, samples)
        # Statistiques par source
        rms = np.sqrt(np.mean(audio**2))
        peak = np.max(np.abs(audio))
        track_dict[f'{source_name}_rms'] = rms
        track_dict[f'{source_name}_peak'] = peak
    
    stats.append(track_dict)

df_stats = pd.DataFrame(stats)
print(df_stats.head())

### Visualisation des caractéristiques des données audio

#### Histogrammes des durées

In [ ]:
plt.figure(figsize=(8,4))
sns.histplot(df_stats['duration_sec'], bins=20, kde=True)
plt.xlabel("Durée (s)")
plt.title("Distribution des durées des morceaux")
plt.show()

#### Mesure de la puissance moyenne d’un signal selon les sources (RMS)

In [ ]:
sources = ['vocals', 'bass', 'drums', 'other', 'accompaniment']  # selon MUSDB
for source in sources:
    if f'{source}_rms' in df_stats.columns:
        plt.figure(figsize=(8,4))
        sns.histplot(df_stats[f'{source}_rms'], bins=20, kde=True)
        plt.xlabel("RMS")
        plt.title(f"Distribution RMS - {source}")
        plt.show()